# 08 — EXP_13 / EXP_14 / EXP_15: Hallucination Error-Type Taxonomy

Classify each wrong-answer question into one of six error categories using `gpt-4o-mini`, then cross-tabulate `category × architecture` to populate Excel **Table 7**.

## The 6 categories (locked at proposal §7.8)

Loaded from [`src/taxonomy/categories.py`](../src/taxonomy/categories.py):

| # | Category | When to use |
|---|---|---|
| 1 | `unsupported_diagnosis` | Predicted diagnosis has no chunk support (and gold IS in chunks) |
| 2 | `unsupported_treatment` | Same for treatment / management questions |
| 3 | `wrong_reasoning_chain` | Chunks support gold; LLM reasoned wrong from them |
| 4 | `partial_evidence_misuse` | Chunk fragment lifted out of context |
| 5 | `option_mismatch` | LLM's prose argues X, outputs Y (output-formatting failure) |
| 6 | `context_omission` | Retrieval missed the chunks needed for gold |

## Scope

- **Surface**: `golden_234` (consistent with Phase 7's RAGAS surface)
- **Questions**: only **wrong-answer** rows per architecture (where `predictions.jsonl[i].is_correct == False`)
- **Architectures**: all 5 (EXP_01 No-RAG · EXP_02 Naive · EXP_03 Sparse · EXP_04 Hybrid · EXP_05 Multi-Hop). ≈ 157 question-arch labels total.
- **Labeller**: `gpt-4o-mini` (JSON-mode) with category definitions baked into the system prompt; disk-cached.
- **Cost**: ~$3 (157 calls × ~$0.02 each). Wall time: ~15 min.

## Stages

1. **Stage A — Smoke** (3 wrong questions × Multi-Hop, ~$0.06)
2. **Stage B — Multi-Hop only** (~23 wrong questions, ~$0.50)
3. **Stage C — All 5 archs** (~157 question-arch labels, ~$3)

Each gated by a boolean at the top of its code cell.

## 1. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

import pandas as pd

from src.data.loaders import load_chunks
from src.taxonomy.labeller import run_taxonomy_batch, DEFAULT_MODEL
from src.taxonomy.analysis import (
    crosstab_category_by_arch, headline_table, load_labels_jsonl,
)
from src.taxonomy.categories import CATEGORY_ORDER

print("OPENAI_API_KEY:", "\u2713" if os.environ.get("OPENAI_API_KEY") else "\u2717")
print("Labeller model:", DEFAULT_MODEL)
print("Categories:", CATEGORY_ORDER)

OUT_DIR = REPO_ROOT / "results" / "exp_14_taxonomy_labels"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_API_KEY: ✓
Labeller model: gpt-4o-mini-2024-07-18
Categories: ('unsupported_diagnosis', 'unsupported_treatment', 'wrong_reasoning_chain', 'partial_evidence_misuse', 'option_mismatch', 'context_omission')


## 2. Helper — assemble wrong-answer rows for an architecture

Joins predictions + retrieval + golden (for question text + options + gold letter) → row dicts the labeller batch consumes.

In [2]:
# Golden 234 question metadata
golden = pd.DataFrame([
    json.loads(line) for line in (REPO_ROOT / "data/processed/golden_ragas_300.jsonl").read_text().splitlines()
])
golden["question_id"] = golden["question_id"].apply(lambda i: f"golden_{int(i):03d}")
GOLDEN = golden.set_index("question_id")

chunks_df = load_chunks().set_index("chunk_id")
print(f"golden_234 metadata: {len(GOLDEN)} rows · chunks parquet: {len(chunks_df):,}")

ARCH_LABEL_TO_PREFIX = {
    "NoRAG": "exp_01_base_llm",
    "Naive": "exp_02_naive_rag",
    "Sparse": "exp_03_sparse_rag",
    "Hybrid": "exp_04_hybrid_rag",
    "MultiHop": "exp_05_multi_hop_rag",
}

def load_wrong_rows(arch_label: str) -> list[dict]:
    """Return labeller-input dicts for all wrong-answer questions on this arch."""
    prefix = ARCH_LABEL_TO_PREFIX[arch_label]
    preds_path = REPO_ROOT / f"results/{prefix}__golden_234/predictions.jsonl"
    retr_path = REPO_ROOT / f"results/{prefix}__golden_234/retrieval.jsonl"
    preds = [json.loads(l) for l in preds_path.read_text().splitlines()]
    retr = {json.loads(l)["question_id"]: json.loads(l) for l in retr_path.read_text().splitlines()}

    rows = []
    for p in preds:
        if p["is_correct"]:
            continue
        qid = p["question_id"]
        if qid not in GOLDEN.index:
            continue
        g = GOLDEN.loc[qid]
        # Retrieved chunks
        rchunks = []
        if qid in retr:
            for cid in retr[qid].get("retrieved_chunk_ids", []):
                if cid in chunks_df.index:
                    rchunks.append({"chunk_id": cid, "text": str(chunks_df.loc[cid, "text"])})
        # Options dict
        opts = g["options"] if isinstance(g["options"], dict) else json.loads(g["options"])
        rows.append({
            "question_id": qid,
            "architecture": arch_label,
            "question": g["question"],
            "options": opts,
            "chunks": rchunks,
            "pred_letter": p.get("pred_letter") or "",
            "pred_text": p.get("raw_response") or "",
            "gold_letter": p.get("gold_letter") or g.get("gold_answer_letter", ""),
        })
    return rows

# Forecast the per-arch wrong counts
print("\n=== Wrong-answer counts per architecture (golden_234) ===")
for arch in ARCH_LABEL_TO_PREFIX:
    rows = load_wrong_rows(arch)
    print(f"  {arch:9s}: {len(rows)} wrong questions")

total = sum(len(load_wrong_rows(a)) for a in ARCH_LABEL_TO_PREFIX)
print(f"\n  Total question-arch labels for Stage C: {total} · estimated cost ≈ ${total * 0.02:.2f}")

golden_234 metadata: 234 rows · chunks parquet: 67,599

=== Wrong-answer counts per architecture (golden_234) ===
  NoRAG    : 23 wrong questions
  Naive    : 35 wrong questions
  Sparse   : 38 wrong questions
  Hybrid   : 38 wrong questions
  MultiHop : 23 wrong questions

  Total question-arch labels for Stage C: 157 · estimated cost ≈ $3.14


## 3. Stage A — Smoke (3 wrong questions × Multi-Hop)

Sanity check: 3 calls to gpt-4o-mini, ~$0.06, ~30 sec. Verify the labeller produces valid JSON, the categories make sense for the cases, and the cache works on re-run.

In [ ]:
RUN_SMOKE = True
RUN_STAGE_B = True
RUN_STAGE_C = False

if RUN_SMOKE:
    smoke_rows = load_wrong_rows("MultiHop")[:3]
    print(f"Smoke surface: {len(smoke_rows)} wrong Multi-Hop questions")
    for r in smoke_rows:
        print(f"  {r['question_id']}: gold={r['gold_letter']} pred={r['pred_letter']} · {r['question'][:80]}…")

    summary = run_taxonomy_batch(
        rows=smoke_rows,
        output_path=OUT_DIR / "smoke_3_multihop.jsonl",
    )
    print("\n=== Smoke summary ===")
    print(json.dumps(summary, indent=2))
else:
    print("RUN_SMOKE = False")

Smoke surface: 3 wrong Multi-Hop questions
  golden_000: gold=B pred=C · A 12-year-old boy who recently immigrated from Namibia is being evaluated for ex…
  golden_007: gold=A pred=D · A 10-year-old boy is brought in by his mother with increasing abdominal pain for…
  golden_023: gold=B pred=D · A 67-year-old woman presents to her physician for a regular checkup. She is a co…


taxonomy:   0%|          | 0/3 [00:00<?, ?it/s]


=== Smoke summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/smoke_3_multihop.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 3,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 0,
  "wall_time_s_this_run": 19.80275583267212
}


## 4. Inspect smoke output

In [4]:
if RUN_SMOKE:
    labels = load_labels_jsonl(OUT_DIR / "smoke_3_multihop.jsonl")
    for _, r in labels.iterrows():
        print(f"--- {r['question_id']} (gold={r['gold_letter']} pred={r['pred_letter']}) ---")
        print(f"  category : {r['category']}")
        print(f"  rationale: {r['rationale']}")
        print(f"  cached={r['was_cached']}  parse_ok={r['parse_ok']}  latency={r['latency_s']:.2f}s")
        print()

    # Acceptance gates for Stage A:
    print("=== Acceptance gates ===")
    print(f"  All parse_ok=True?  {bool(labels['parse_ok'].all())}")
    print(f"  All category in CATEGORY_ORDER?  {bool(labels['category'].isin(CATEGORY_ORDER).all())}")
    print(f"  Distinct categories used: {sorted(labels['category'].unique().tolist())}")
    print("\nIf both gates pass and the rationales are plausible, flip RUN_STAGE_B = True and re-run.")

--- golden_000 (gold=B pred=C) ---
  category : wrong_reasoning_chain
  rationale: The retrieved evidence supports that the boy's symptoms are due to a Type II hypersensitivity reaction related to rheumatic fever, as indicated in chunk [6], which describes the immune response to streptococcal antigens causing tissue damage. However, the model incorrectly inferred Type III hypersensitivity as the mechanism.
  cached=False  parse_ok=True  latency=10.77s

--- golden_007 (gold=A pred=D) ---
  category : wrong_reasoning_chain
  rationale: The retrieved evidence supports the diagnosis of Burkitt lymphoma, particularly with the mention of 'anemia, FTT, and fever' and the characteristics of tumors in children, which align with the features of Burkitt lymphoma [4]. However, the model incorrectly concluded that the diagnosis was Wilms tumor (D), despite acknowledging relevant evidence for the correct answer (A).
  cached=False  parse_ok=True  latency=3.37s

--- golden_023 (gold=B pred=D) ---
  c

## 5. Stage B — Multi-Hop only (all wrong questions)

Cost: ~$0.50 (23 × ~$0.02). ~3 min wall time.

In [5]:
RUN_STAGE_B = True
if RUN_STAGE_B:
    rows_b = load_wrong_rows("MultiHop")
    print(f"Stage B: {len(rows_b)} wrong Multi-Hop questions")
    summary_b = run_taxonomy_batch(
        rows=rows_b,
        output_path=OUT_DIR / "stage_b_multihop_wrong.jsonl",
    )
    print("\n=== Stage B summary ===")
    print(json.dumps(summary_b, indent=2))

    labels_b = load_labels_jsonl(OUT_DIR / "stage_b_multihop_wrong.jsonl")
    print("\n=== Multi-Hop wrong-answer category distribution ===")
    print(labels_b["category"].value_counts().reindex(list(CATEGORY_ORDER), fill_value=0))
else:
    print("RUN_STAGE_B = False — flip after Stage A gates pass")

Stage B: 23 wrong Multi-Hop questions


taxonomy:   0%|          | 0/23 [00:00<?, ?it/s]


=== Stage B summary ===
{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_b_multihop_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 23,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 3,
  "wall_time_s_this_run": 72.74004411697388
}

=== Multi-Hop wrong-answer category distribution ===
category
unsupported_diagnosis       2
unsupported_treatment      10
wrong_reasoning_chain      11
partial_evidence_misuse     0
option_mismatch             0
context_omission            0
Name: count, dtype: int64


## 6. Stage C — All 5 architectures (full Table 7)

Cost: ~$3 (Stage B's Multi-Hop calls hit cache; only the other 4 archs make fresh calls). Wall time: ~15 min.

In [7]:
RUN_STAGE_C = True
if RUN_STAGE_C:
    all_summaries = {}
    for arch in ARCH_LABEL_TO_PREFIX:
        rows = load_wrong_rows(arch)
        print(f"\n--- {arch} ({len(rows)} wrong questions) ---")
        s = run_taxonomy_batch(
            rows=rows,
            output_path=OUT_DIR / f"stage_c_{arch.lower()}_wrong.jsonl",
        )
        all_summaries[arch] = s
        print(json.dumps(s, indent=2))
else:
    print("RUN_STAGE_C = False — flip after Stage B is clean")


--- NoRAG (23 wrong questions) ---


taxonomy:   0%|          | 0/23 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_c_norag_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 23,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 0,
  "wall_time_s_this_run": 58.559386253356934
}

--- Naive (35 wrong questions) ---


taxonomy:   0%|          | 0/35 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_c_naive_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 35,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 0,
  "wall_time_s_this_run": 103.00996136665344
}

--- Sparse (38 wrong questions) ---


...

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_c_sparse_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 38,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 0,
  "wall_time_s_this_run": 130.1284019947052
}

--- Hybrid (38 wrong questions) ---


taxonomy:   0%|          | 0/38 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_c_hybrid_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 38,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 0,
  "wall_time_s_this_run": 126.74554705619812
}

--- MultiHop (23 wrong questions) ---


taxonomy:   0%|          | 0/23 [00:00<?, ?it/s]

{
  "output_path": "/Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_14_taxonomy_labels/stage_c_multihop_wrong.jsonl",
  "model": "gpt-4o-mini-2024-07-18",
  "n_rows_written_this_run": 23,
  "n_rows_already_done": 0,
  "n_parse_failures": 0,
  "n_cache_hits_this_run": 23,
  "wall_time_s_this_run": 0.024927854537963867
}


## 7. EXP_15 — Cross-tab error_type × architecture (Excel Table 7)

Aggregates whatever Stage B / C files are on disk. Produces both counts (paste-ready) and within-architecture proportions (interpretive).

In [8]:
all_files = sorted(OUT_DIR.glob("stage_*.jsonl"))
if not all_files:
    print("No stage_*.jsonl files yet — run Stage B or C first.")
else:
    dfs = [load_labels_jsonl(p) for p in all_files]
    df_all = pd.concat(dfs, ignore_index=True)
    # Dedup on (question_id, architecture) — multiple stages may overlap (e.g. Multi-Hop appears in B + C)
    df_all = df_all.drop_duplicates(subset=["question_id", "architecture"], keep="last")
    print(f"Aggregated label rows: {len(df_all)} unique (qid, arch) labels")
    print(f"Parse failures: {(~df_all['parse_ok']).sum()}")

    print("\n=== Table 7 — counts ===")
    ct_counts = crosstab_category_by_arch(df_all)
    print(ct_counts.to_string())

    print("\n=== Table 7 — within-architecture proportions ===")
    ct_props = crosstab_category_by_arch(df_all, normalize="columns")
    print(ct_props.to_string())

    # Write paste-ready CSV for the workbook
    ANALYSIS_DIR = REPO_ROOT / "results" / "exp_15_taxonomy_analysis"
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    ct_counts.to_csv(ANALYSIS_DIR / "table7_counts.csv")
    ct_props.to_csv(ANALYSIS_DIR / "table7_proportions.csv")
    print(f"\n✓ Wrote {ANALYSIS_DIR / 'table7_counts.csv'}")
    print(f"✓ Wrote {ANALYSIS_DIR / 'table7_proportions.csv'}")

    print("\n=== Per-architecture headline ===")
    print(headline_table(df_all).to_string(index=False))

Aggregated label rows: 157 unique (qid, arch) labels
Parse failures: 0

=== Table 7 — counts ===
architecture             Hybrid  MultiHop  Naive  NoRAG  Sparse
category                                                       
unsupported_diagnosis         4         2      6      0       4
unsupported_treatment        17        10      8      0      12
wrong_reasoning_chain        17        11     21      0      22
partial_evidence_misuse       0         0      0      0       0
option_mismatch               0         0      0      0       0
context_omission              0         0      0     23       0

=== Table 7 — within-architecture proportions ===
architecture             Hybrid  MultiHop   Naive  NoRAG  Sparse
category                                                        
unsupported_diagnosis    0.1053    0.0870  0.1714    0.0  0.1053
unsupported_treatment    0.4474    0.4348  0.2286    0.0  0.3158
wrong_reasoning_chain    0.4474    0.4783  0.6000    0.0  0.5789
partial_evidenc

---

**Next.** Phase 9 — EXP_16 final synthesis. Weighted ranking of all architectures (0.25 Accuracy + 0.25 Faithfulness + 0.20 Retrieval + 0.15 Safety + 0.10 Explainability + 0.05 Latency) → deployment recommendation. Pure aggregation; $0; ~30 min.